In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq",
    temperature=0.7,
    max_tokens=1000,
    max_retries=2,
)

model


c:\Users\yogeshkannah\Music\AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000016895000CD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000016895119650>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), max_tokens=1000)

## Chatbot with LangGraph

StageGraph - Which help statemanagement of the chatbot
START and END are special nodes

In [22]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


In [23]:
class State(TypedDict):
    # Messages are the type List, the add_messages function 
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting the)
    messages : Annotated[list, add_messages]

graph_builder = StateGraph(State)
graph_builder

In [31]:
def chatbot(state: State):
    print("\nINSIDE NODE - Incoming State:")
    for msg in state["messages"]:
        print(f"INNER NODE {msg.type}: {msg.content}")

    response = model.invoke(state["messages"])
    print("\nINSIDE NODE - Outgoing State:")
    print(response)
    return {"messages": [response]}

In [25]:
graph_builder.add_node("chatbot", chatbot)

In [26]:
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)


In [27]:
graph = graph_builder.compile()

In [33]:
while True:
    user_input = input("User: ")
    if user_input.lower() == "exit":
        break
    print("User INPUT: ", user_input)
    for event in graph.stream({"messages": [{"role": "user", "content": user_input}]},stream_mode="values"):
        print("Event: ", event)
        print(event.values())
        
        for value in event["messages"]:
            print("\nAssistant: ", value.content)

User INPUT:  hi
Event:  {'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='80012048-cb66-4738-b889-6b4561c04e18')]}
dict_values([[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='80012048-cb66-4738-b889-6b4561c04e18')]])

Assistant:  hi

INSIDE NODE - Incoming State:
human: hi

INSIDE NODE - Outgoing State:
content='How can I assist you today?' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 36, 'total_tokens': 44, 'completion_time': 0.006192288, 'completion_tokens_details': None, 'prompt_time': 0.002115419, 'prompt_tokens_details': None, 'queue_time': 0.045754227, 'total_time': 0.008307707}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019c9ebe-fc26-7273-bad6-581408caeb31-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_t